# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rasheed-hammad/machine-learning-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Random Forest classifier because the refresh-review decision can depend on combinations of search visibility, content staleness, ranking position, and content metadata. The W04 baseline used a simple transparent rule based mainly on staleness and impressions, while the Random Forest can learn nonlinear interactions among several leakage-safe features. I will judge the model by whether it improves the same ranking metric against the W04 baseline, not by model complexity alone.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("HF_TOKEN loaded:", HF_TOKEN is not None)
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

HF_TOKEN loaded: True
Token length: 37


In [2]:
from huggingface_hub import whoami

user_info = whoami(token=HF_TOKEN)

print("Hugging Face authentication successful.")
print("Username:", user_info.get("name"))

Hugging Face authentication successful.
Username: hammadrasheed


In [3]:
# Section 1 — Load and inspect modeling data

import os
import pandas as pd
import duckdb
from google.colab import userdata
from huggingface_hub import hf_hub_download

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_Token")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

con = duckdb.connect()

repo_id = "FlyRank/internship-warehouse"
repo_type = "dataset"

march_file = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type=repo_type,
    token=HF_TOKEN
)

april_file = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type=repo_type,
    token=HF_TOKEN
)

print("March file:", march_file)
print("April file:", april_file)

march_check = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM read_parquet('{march_file}')
""").fetchdf()

april_check = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM read_parquet('{april_file}')
""").fetchdf()

print("\nMarch:")
display(march_check)

print("\nApril:")
display(april_check)

HF_TOKEN loaded: True
March file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
April file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


March:


,min_date,max_date,rows,clients,content_items
0,2026-03-01,2026-03-31,9841378,55,331437



April:


,min_date,max_date,rows,clients,content_items
0,2026-04-01,2026-04-30,10424730,61,362172


In [4]:
# Section 1 — Build March features and April outcome

march = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions
    FROM read_parquet('{march_file}')
""").fetchdf()

april = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('{april_file}')
""").fetchdf()

print("March rows loaded:", len(march))
print("April rows loaded:", len(april))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows loaded: 9841378
April rows loaded: 10424730


In [5]:
# Load content metadata for modeling

content_file = hf_hub_download(
    repo_id=repo_id,
    filename="dim_content.parquet",
    repo_type=repo_type,
    token=HF_TOKEN
)

content = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        content_type,
        search_volume,
        competition,
        backlinks,
        word_count,
        is_published,
        is_deleted
    FROM read_parquet('{content_file}')
""").fetchdf()

print("Content metadata rows:", len(content))
display(content.head())

Content metadata rows: 519606


,client_hash_id,content_hash_id,content_updated_date,content_type,search_volume,competition,backlinks,word_count,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,2026-07-01,keyword article,30,0.91,16,2555,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,2026-07-01,keyword article,10,0.00,0,2430,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,2026-07-01,keyword article,480,0.36,169,2645,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,2026-06-15,keyword article,0,0.00,0,2522,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,2026-06-01,keyword article,2400,0.70,52,2552,True,False


In [6]:
# Section 1 — Create page-level March features

march_page = (
    march
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "median"),
        march_pageviews=("ga4_pageviews", "sum"),
        march_sessions=("ga4_sessions", "sum"),
        march_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)

march_page["march_ctr"] = (
    march_page["march_clicks"] / march_page["march_impressions"]
).where(march_page["march_impressions"] > 0)

print("March page-level rows:", len(march_page))
display(march_page.head())

March page-level rows: 331437


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_pageviews,march_sessions,march_engaged_sessions,march_ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0,0,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0,0,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0,0,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0,0,0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0,0,0,NaN


In [7]:
# Section 1 — Create future April outcome

april_page = (
    april
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum"),
        april_avg_position=("gsc_avg_position", "median")
    )
)

print("April page-level rows:", len(april_page))
display(april_page.head())

April page-level rows: 362172


,client_hash_id,content_hash_id,april_impressions,april_clicks,april_avg_position
0,client_06d356715a8ff3b6,content_0059a4d4195810c9,873,2,7.746529
1,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,634,1,9.697637
2,client_06d356715a8ff3b6,content_0153b7dedc3fc40d,640,2,7.838333
3,client_06d356715a8ff3b6,content_0241f6a890063db0,85,1,3.573096
4,client_06d356715a8ff3b6,content_045f673b3d3c18a4,153,2,3.980769


In [8]:
# Section 1 — Join past features to future outcome

model_df = march_page.merge(
    april_page,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Pages present in both March and April:", len(model_df))

Pages present in both March and April: 331436


In [9]:
# Section 1 — Add leakage-safe content metadata

model_df = model_df.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Keep metadata that was already available by the end of March.
model_df = model_df[
    (model_df["content_updated_date"].notna()) &
    (model_df["content_updated_date"] <= pd.Timestamp("2026-03-31")) &
    (model_df["is_published"] == True) &
    (model_df["is_deleted"] == False)
].copy()

model_df["days_since_update"] = (
    pd.Timestamp("2026-03-31") -
    model_df["content_updated_date"]
).dt.days

print("Model rows after metadata filtering:", len(model_df))

Model rows after metadata filtering: 37229


In [10]:
# Section 1 — Define the future decline target

model_df["impression_change_pct"] = (
    (model_df["april_impressions"] - model_df["march_impressions"])
    / model_df["march_impressions"]
) * 100

model_df["target_declined"] = (
    (model_df["march_impressions"] >= 100) &
    (model_df["impression_change_pct"] <= -30)
).astype(int)

print("Target distribution:")
display(
    model_df["target_declined"]
    .value_counts()
    .rename_axis("target_declined")
    .reset_index(name="n")
)

print("\nTarget rate:",
      round(model_df["target_declined"].mean() * 100, 2), "%")

Target distribution:


,target_declined,n
0,0,28096
1,1,9133



Target rate: 24.53 %


In [11]:
# Check April GSC availability before finalizing the target

april_availability = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS april_days,
        SUM(CASE WHEN gsc_data_available = TRUE THEN 1 ELSE 0 END) AS gsc_available_days,
        SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS gsc_unavailable_days
    FROM read_parquet('{april_file}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

model_df = model_df.merge(
    april_availability,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Pages with April availability information:", len(model_df))

display(
    model_df[
        ["april_days", "gsc_available_days", "gsc_unavailable_days"]
    ].describe()
)

Pages with April availability information: 37229


,april_days,gsc_available_days,gsc_unavailable_days
count,37229.0,37229.000000,37229.000000
mean,30.0,16.044132,13.955868
std,0.0,13.408077,13.408077
min,30.0,0.000000,0.000000
25%,30.0,0.000000,0.000000
50%,30.0,20.000000,10.000000
75%,30.0,30.000000,30.000000
max,30.0,30.000000,30.000000


In [12]:
print("Pages with all 30 April days available:",
      (model_df["gsc_available_days"] == 30).sum())

print("Pages with at least 20 April days available:",
      (model_df["gsc_available_days"] >= 20).sum())

print("Pages with fewer than 20 April days available:",
      (model_df["gsc_available_days"] < 20).sum())

Pages with all 30 April days available: 12660
Pages with at least 20 April days available: 18750
Pages with fewer than 20 April days available: 18479


In [13]:
eligible = model_df["march_impressions"] >= 100

print("Eligible pages with March impressions >= 100:", eligible.sum())

print(
    "Decline rate among eligible pages:",
    round(model_df.loc[eligible, "target_declined"].mean() * 100, 2),
    "%"
)

Eligible pages with March impressions >= 100: 18079
Decline rate among eligible pages: 50.52 %


In [14]:
# Final W05 modeling population and future-decline target

model_df["eligible_for_model"] = (
    (model_df["march_impressions"] >= 100) &
    (model_df["gsc_available_days"] >= 20)
)

model_df["target_declined"] = (
    model_df["eligible_for_model"] &
    (model_df["impression_change_pct"] <= -30)
).astype(int)

model_ready = model_df[
    model_df["eligible_for_model"]
].copy()

print("Final modeling pages:", len(model_ready))

print("\nTarget distribution:")
display(
    model_ready["target_declined"]
    .value_counts()
    .rename_axis("target_declined")
    .reset_index(name="n")
)

print(
    "\nFuture-decline proxy rate:",
    round(model_ready["target_declined"].mean() * 100, 2),
    "%"
)

Final modeling pages: 16957

Target distribution:


,target_declined,n
0,0,8919
1,1,8038



Future-decline proxy rate: 47.4 %


In [15]:
print("Final modeling population checks:")

print(
    "Minimum March impressions:",
    model_ready["march_impressions"].min()
)

print(
    "Minimum April GSC available days:",
    model_ready["gsc_available_days"].min()
)

print(
    "Pages with target = 1:",
    model_ready["target_declined"].sum()
)

print(
    "Pages with target = 0:",
    (model_ready["target_declined"] == 0).sum()
)

Final modeling population checks:
Minimum March impressions: 100
Minimum April GSC available days: 20.0
Pages with target = 1: 8038
Pages with target = 0: 8919


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-level holdout split so that pages from the same client do not appear in both training and test sets. Approximately 80% of clients will be used for training and 20% for testing, with a fixed random seed for reproducibility. The test clients will remain unseen during model fitting and will be used for the final comparison between the Random Forest and the W04 baseline. This is intended to provide a more realistic estimate of how the ranking approach may perform on unseen clients.


In [16]:
# Section 2 — Client-level train/test split

from sklearn.model_selection import train_test_split

clients = model_ready["client_hash_id"].unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = model_ready[
    model_ready["client_hash_id"].isin(train_clients)
].copy()

test_df = model_ready[
    model_ready["client_hash_id"].isin(test_clients)
].copy()

print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nTraining pages:", len(train_df))
print("Test pages:", len(test_df))

Total clients: 26
Training clients: 20
Test clients: 6

Training pages: 16897
Test pages: 60


In [17]:
# Verify that no client appears in both train and test

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("PASS: No client appears in both train and test.")
else:
    print("FAIL: Client leakage detected.")

Client overlap: 0
PASS: No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The model will use March-only information to predict the April future-decline proxy. I will use March impressions, clicks, CTR, average position, GA4 activity, content staleness, and content type. April performance is used only to construct the target and will not be used as a model feature. I will also exclude the starter outcome fields such as trend_direction, trend_pct, and is_declining_label. The model will be evaluated on the held-out clients and compared with the W04 rule using the same Precision@20 ranking metric.

In [18]:
# Section 3 — Select leakage-safe March features

feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update",
    "content_type"
]

target_column = "target_declined"

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("Features:")
for feature in feature_columns:
    print("-", feature)

print("\nTraining rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training target rate:", round(y_train.mean() * 100, 2), "%")
print("Test target rate:", round(y_test.mean() * 100, 2), "%")

Features:
- march_impressions
- march_clicks
- march_ctr
- march_avg_position
- march_pageviews
- march_sessions
- march_engaged_sessions
- days_since_update
- content_type

Training rows: 16897
Test rows: 60
Training target rate: 47.45 %
Test target rate: 35.0 %


In [19]:
# Train Random Forest

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update"
]

categorical_features = [
    "content_type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

model_pipeline.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [20]:
# Generate probability scores for the test pages

test_df = test_df.copy()

test_df["model_score"] = model_pipeline.predict_proba(X_test)[:, 1]

test_ranked = test_df.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

test_ranked["model_rank"] = range(1, len(test_ranked) + 1)

print("Model predictions generated:", len(test_ranked))

display(
    test_ranked[
        [
            "model_rank",
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "target_declined",
            "march_impressions",
            "march_avg_position",
            "days_since_update"
        ]
    ].head(20)
)

Model predictions generated: 60


,model_rank,client_hash_id,content_hash_id,model_score,target_declined,march_impressions,march_avg_position,days_since_update
0,1,client_3f0ce4d44fe94f3d,content_dfa55ff1b1c7ab00,0.800000,0,736,2.740741,35
1,2,client_3f0ce4d44fe94f3d,content_e566dd54de604978,0.786667,1,365,2.000000,35
2,3,client_3f0ce4d44fe94f3d,content_6c589a6c7158482e,0.743333,0,293,3.000000,35
3,4,client_0797ff3a1fc9a6a5,content_27f8100281413b37,0.723333,0,464,8.845202,34
4,5,client_3f0ce4d44fe94f3d,content_61f8fec9c4ce645f,0.716667,1,3880,2.739130,35
5,6,client_3f0ce4d44fe94f3d,content_a938a01820c92489,0.616667,1,425,0.770833,35
6,7,client_3f0ce4d44fe94f3d,content_302ec4da585c050e,0.603333,0,638,5.666667,35
7,8,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,0.563333,0,774,12.214286,34
8,9,client_f623b01661d4bfe4,content_a47fd3545d1a68c6,0.563333,1,4925,8.166667,81
9,10,client_3f0ce4d44fe94f3d,content_0c6745c92d4f3415,0.560000,0,136,3.400000,35


In [21]:
# Recreate W04 baseline on the same held-out test pages

test_ranked["baseline_stale"] = (
    test_ranked["days_since_update"] >= 180
)

test_ranked["baseline_visible"] = (
    test_ranked["march_impressions"] >= 500
)

test_ranked["baseline_score"] = (
    test_ranked["baseline_stale"].astype(int)
    * test_ranked["baseline_visible"].astype(int)
    * test_ranked["march_impressions"]
)

baseline_ranked = test_ranked.sort_values(
    ["baseline_score", "march_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_ranked["baseline_rank"] = range(
    1,
    len(baseline_ranked) + 1
)

print("Baseline ranking created:", len(baseline_ranked))

display(
    baseline_ranked[
        [
            "baseline_rank",
            "client_hash_id",
            "content_hash_id",
            "baseline_score",
            "target_declined",
            "march_impressions",
            "days_since_update"
        ]
    ].head(20)
)

Baseline ranking created: 60


,baseline_rank,client_hash_id,content_hash_id,baseline_score,target_declined,march_impressions,days_since_update
0,1,client_f623b01661d4bfe4,content_b5bef91d1b43e20c,0,1,14738,81
1,2,client_f623b01661d4bfe4,content_a47fd3545d1a68c6,0,1,4925,81
2,3,client_3f0ce4d44fe94f3d,content_61f8fec9c4ce645f,0,1,3880,35
3,4,client_0797ff3a1fc9a6a5,content_a7a5f0d74ec03ce8,0,0,3547,34
4,5,client_f623b01661d4bfe4,content_4ae3607e3e51b9ba,0,0,2930,81
5,6,client_0797ff3a1fc9a6a5,content_be06033d30b49299,0,1,2092,34
6,7,client_0797ff3a1fc9a6a5,content_fa84e03e3d5e2fa2,0,0,1935,34
7,8,client_f623b01661d4bfe4,content_beba734159003307,0,0,1542,81
8,9,client_3f0ce4d44fe94f3d,content_58354e9e89956da2,0,1,1393,35
9,10,client_3ffa76342f366962,content_25aac3e5439711c6,0,0,1327,141


In [22]:
# Compare Random Forest and W04 baseline using Precision@20

k = 20

model_top20 = test_ranked.head(k)
baseline_top20 = baseline_ranked.head(k)

model_precision_at_20 = model_top20["target_declined"].mean()
baseline_precision_at_20 = baseline_top20["target_declined"].mean()

comparison = pd.DataFrame({
    "method": [
        "W04 baseline",
        "Random Forest"
    ],
    "precision_at_20": [
        baseline_precision_at_20,
        model_precision_at_20
    ],
    "positive_pages_in_top20": [
        int(baseline_top20["target_declined"].sum()),
        int(model_top20["target_declined"].sum())
    ]
})

display(comparison)

,method,precision_at_20,positive_pages_in_top20
0,W04 baseline,0.35,7
1,Random Forest,0.30,6


In [23]:
# Inspect Random Forest top-20 outcomes

display(
    model_top20[
        [
            "model_rank",
            "content_hash_id",
            "model_score",
            "target_declined",
            "march_impressions",
            "march_avg_position",
            "days_since_update"
        ]
    ]
)

,model_rank,content_hash_id,model_score,target_declined,march_impressions,march_avg_position,days_since_update
0,1,content_dfa55ff1b1c7ab00,0.800000,0,736,2.740741,35
1,2,content_e566dd54de604978,0.786667,1,365,2.000000,35
2,3,content_6c589a6c7158482e,0.743333,0,293,3.000000,35
3,4,content_27f8100281413b37,0.723333,0,464,8.845202,34
4,5,content_61f8fec9c4ce645f,0.716667,1,3880,2.739130,35
5,6,content_a938a01820c92489,0.616667,1,425,0.770833,35
6,7,content_302ec4da585c050e,0.603333,0,638,5.666667,35
7,8,content_37952b007ab057b3,0.563333,0,774,12.214286,34
8,9,content_a47fd3545d1a68c6,0.563333,1,4925,8.166667,81
9,10,content_0c6745c92d4f3415,0.560000,0,136,3.400000,35


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest achieved Precision@20 of 0.30 on the held-out clients, compared with 0.35 for the W04 baseline. Therefore, the model did not improve the baseline on this test split. This result suggests that the additional nonlinear modeling did not provide a clear ranking advantage for this particular future-decline proxy and small held-out test set.

The Random Forest still identified 6 declining pages in its top 20, so its predictions contain useful signal, but the baseline identified 7. The test set contains only 60 pages, so the Precision@20 estimate is sensitive to individual pages and should be treated as directional rather than definitive.

The model's errors are also useful for understanding the decision problem. False positives represent pages that the model ranked highly but did not meet the future-decline proxy, while false negatives represent declining pages that were ranked lower. The model should therefore be treated as decision support rather than an automatic refresh decision.


The Random Forest's top-20 queue contained 6 true positives and 14 false positives, giving Precision@20 of 0.30. Fifteen additional declining pages were ranked below the top 20, showing that the model missed a meaningful number of future-decline cases.

The feature importance results show that March average position (0.318) and March impressions (0.294) were the two strongest model features, followed by March CTR (0.140), pageviews (0.084), and sessions (0.081). Days since update had relatively low importance (0.013), suggesting that the learned model relied more heavily on current search-performance signals than content staleness.

The W04 baseline achieved Precision@20 of 0.35 compared with 0.30 for the Random Forest. Therefore, the Random Forest did not outperform the simpler baseline on this held-out test. The result does not justify replacing the baseline with the model based on this experiment alone.

The comparison should also be interpreted cautiously because the test set contained only 60 eligible pages from 6 held-out clients. Precision@20 therefore represents a small-sample, directional evaluation. The future-decline label is also a proxy based on April impressions rather than a direct measurement of whether a page needed a content refresh.


In [24]:
# Random Forest top-20 error analysis

model_top20 = test_ranked.head(20).copy()

model_top20["error_type"] = model_top20["target_declined"].map({
    1: "true_positive",
    0: "false_positive"
})

print("Random Forest top-20:")
display(
    model_top20[
        [
            "model_rank",
            "content_hash_id",
            "model_score",
            "target_declined",
            "error_type",
            "march_impressions",
            "march_avg_position",
            "days_since_update"
        ]
    ]
)

print("\nTop-20 error counts:")
display(
    model_top20["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="n")
)

Random Forest top-20:


,model_rank,content_hash_id,model_score,target_declined,error_type,march_impressions,march_avg_position,days_since_update
0,1,content_dfa55ff1b1c7ab00,0.800000,0,false_positive,736,2.740741,35
1,2,content_e566dd54de604978,0.786667,1,true_positive,365,2.000000,35
2,3,content_6c589a6c7158482e,0.743333,0,false_positive,293,3.000000,35
3,4,content_27f8100281413b37,0.723333,0,false_positive,464,8.845202,34
4,5,content_61f8fec9c4ce645f,0.716667,1,true_positive,3880,2.739130,35
5,6,content_a938a01820c92489,0.616667,1,true_positive,425,0.770833,35
6,7,content_302ec4da585c050e,0.603333,0,false_positive,638,5.666667,35
7,8,content_37952b007ab057b3,0.563333,0,false_positive,774,12.214286,34
8,9,content_a47fd3545d1a68c6,0.563333,1,true_positive,4925,8.166667,81
9,10,content_0c6745c92d4f3415,0.560000,0,false_positive,136,3.400000,35



Top-20 error counts:


,error_type,n
0,false_positive,14
1,true_positive,6


In [25]:
# Declining pages that the model ranked outside the top 20

missed_declines = test_ranked[
    (test_ranked["target_declined"] == 1) &
    (test_ranked["model_rank"] > 20)
].copy()

print("Declining pages missed by model top-20:", len(missed_declines))

display(
    missed_declines[
        [
            "model_rank",
            "content_hash_id",
            "model_score",
            "march_impressions",
            "march_avg_position",
            "days_since_update"
        ]
    ].head(20)
)

Declining pages missed by model top-20: 15


,model_rank,content_hash_id,model_score,march_impressions,march_avg_position,days_since_update
20,21,content_619be04c28631444,0.433333,138,6.615385,35
22,23,content_ebab0bfa86ef4345,0.423333,142,34.160714,81
23,24,content_9eeba1f980fd7436,0.406667,587,1.920000,35
24,25,content_be06033d30b49299,0.403333,2092,51.984848,34
25,26,content_bdc656fc8f037ac0,0.393333,190,4.500000,141
28,29,content_7d8c5d6c5fae7bd4,0.376667,599,51.956522,81
29,30,content_58354e9e89956da2,0.376667,1393,27.454545,35
31,32,content_d58f9834ef48e26e,0.346667,247,4.958333,34
34,35,content_fc1c7ca27b62e70d,0.310000,679,4.833333,35
37,38,content_9abace70356e4fe5,0.266667,879,66.750000,81


In [26]:
# Random Forest feature importance

feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = model_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

print("Top model features:")

display(
    importance_df.head(15)
)

Top model features:


,feature,importance
3,numeric__march_avg_position,0.317975
0,numeric__march_impressions,0.293700
2,numeric__march_ctr,0.139771
4,numeric__march_pageviews,0.084470
5,numeric__march_sessions,0.080817
1,numeric__march_clicks,0.047466
6,numeric__march_engaged_sessions,0.014913
7,numeric__days_since_update,0.012928
10,categorical__content_type_keyword article,0.002853
9,categorical__content_type_feedly article,0.002576
